In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark Filtering Example") \
    .getOrCreate()

In [0]:
from pyspark.sql.functions import *

Extract Data - Read the customer and sales datasets from CSV

In [0]:
customers = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("/Workspace/Users/manthenapoojith@gmail.com/Customers(P3).csv")

sales = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("/Workspace/Users/manthenapoojith@gmail.com/Sales(P3).csv")

Display Datasets

In [0]:
display(customers)

display(sales)

customer_id,name,city,age
C101,Aadhvik Rao,Bengaluru,24
C102,Ishani Mehta,Mumbai,31
C103,Vihaan Kapoor,Delhi,29
C104,Myra Nair,Hyderabad,35
C105,Rudra Sen,Kolkata,27
C106,Kiara Thomas,Chennai,40
C107,Reyansh Gupta,Pune,26
C108,Anika Reddy,Hyderabad,38
C109,Vivaan Joseph,Kochi,23
C110,Tara Malhotra,Jaipur,33


order_id,customer_id,order_date,order_amount
O001,C101,2025-01-01,2500
O002,C102,2025-01-01,4000
O003,C101,2025-01-02,3500
O004,C103,2025-01-02,2000
O005,C104,2025-01-03,6000
O006,C104,2025-01-03,3000
O007,C106,2025-01-04,4500
O008,C107,2025-01-04,7000
O009,C107,2025-01-04,1000
O010,C108,2025-01-05,8000


In [0]:
customers.printSchema()

sales.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_amount: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col,isnan,when,count

customers.select(
[
count(
when(
col(c).isNull(),
c
)
).alias(c)
for c in customers.columns
]
).show()


sales.select(
[
count(
when(
col(c).isNull(),
c
)
).alias(c)
for c in sales.columns
]
).show()

+-----------+----+----+---+
|customer_id|name|city|age|
+-----------+----+----+---+
|          0|   0|   1|  1|
+-----------+----+----+---+

+--------+-----------+----------+------------+
|order_id|customer_id|order_date|order_amount|
+--------+-----------+----------+------------+
|       0|          1|         0|           1|
+--------+-----------+----------+------------+



In [0]:
customers = customers.dropna(
subset=[
"customer_id",
"city"
]
)

sales = sales.dropna(
subset=[
"customer_id",
"order_amount"
]
)

In [0]:
sales = sales.filter(
col("order_amount")>0
)

In [0]:
display(customers)

display(sales)

customer_id,name,city,age
C101,Aadhvik Rao,Bengaluru,24
C102,Ishani Mehta,Mumbai,31
C103,Vihaan Kapoor,Delhi,29
C104,Myra Nair,Hyderabad,35
C105,Rudra Sen,Kolkata,27
C106,Kiara Thomas,Chennai,40
C107,Reyansh Gupta,Pune,26
C108,Anika Reddy,Hyderabad,38
C109,Vivaan Joseph,Kochi,23
C110,Tara Malhotra,Jaipur,33


order_id,customer_id,order_date,order_amount
O001,C101,2025-01-01,2500
O002,C102,2025-01-01,4000
O003,C101,2025-01-02,3500
O004,C103,2025-01-02,2000
O005,C104,2025-01-03,6000
O006,C104,2025-01-03,3000
O007,C106,2025-01-04,4500
O008,C107,2025-01-04,7000
O009,C107,2025-01-04,1000
O010,C108,2025-01-05,8000


In [0]:
sales = sales.filter(
col("order_amount")>0
)

In [0]:
customer_json = spark.read.json(
"/Workspace/Users/manthenapoojith@gmail.com/Customers(P3).json"
)

display(customer_json)

age,city,customer_id,name
24,Bengaluru,C101,Aadhvik Rao
31,Mumbai,C102,Ishani Mehta
29,Delhi,C103,Vihaan Kapoor
35,Hyderabad,C104,Myra Nair
27,Kolkata,C105,Rudra Sen
40,Chennai,C106,Kiara Thomas
26,Pune,C107,Reyansh Gupta
38,Hyderabad,C108,Anika Reddy
23,Kochi,C109,Vivaan Joseph
33,Jaipur,C110,Tara Malhotra


Final Report Table

In [0]:
report = customers.join(
    sales,
    "customer_id"
).groupBy(
    "customer_id",
    "name",
    "city"
).agg(
    sum("order_amount").alias("total_spend")
)

Exercise **1**
Read sales data -> clean nulls -> calculate daily sales

In [0]:
customers.createOrReplaceTempView("customers")
sales.createOrReplaceTempView("sales")

In [0]:
%sql
SELECT
order_date,
SUM(order_amount)
FROM sales
GROUP BY order_date;

order_date,SUM(order_amount)
2025-01-04,12500
2025-01-05,12000
2025-01-01,6500
2025-01-03,9000
2025-01-06,5000
2025-01-02,5500


In [0]:
display(

sales.groupBy(
"order_date"
).agg(
sum("order_amount")
.alias("daily_sales")
)

)

order_date,daily_sales
2025-01-04,12500
2025-01-05,12000
2025-01-01,6500
2025-01-03,9000
2025-01-06,5000
2025-01-02,5500


2. Read customer data -> clean invalid rows -> city-wise revenue

In [0]:
%sql
SELECT
city,
SUM(order_amount)
FROM customers c
JOIN sales s
ON c.customer_id=s.customer_id
GROUP BY city;

city,SUM(order_amount)
Delhi,2000
Chennai,4500
Jaipur,5000
Hyderabad,21000
Pune,8000
Mumbai,4000
Bengaluru,6000


In [0]:
display(

customers.join(
sales,
"customer_id"
)

.groupBy(
"city"
)

.agg(
sum("order_amount")
.alias("city_revenue")
)

)

city,city_revenue
Delhi,2000
Chennai,4500
Jaipur,5000
Hyderabad,21000
Pune,8000
Mumbai,4000
Bengaluru,6000


3. Find repeat customers (>2 orders)

In [0]:
%sql
SELECT
customer_id,
COUNT(*)
FROM sales
GROUP BY customer_id
HAVING COUNT(*)>2;

customer_id,COUNT(*)
C108,3


In [0]:
display(

sales.groupBy(
"customer_id"
)

.agg(
count("*")
.alias("orders")
)

.filter(
col("orders")>2
)

)

customer_id,orders
C108,3


4. Find highest spending customer in each city

In [0]:
report.createOrReplaceTempView("reporting")

In [0]:
%sql
SELECT city,
MAX(total_spend)
FROM reporting
GROUP BY city;

city,MAX(total_spend)
Delhi,2000
Hyderabad,12000
Bengaluru,6000
Pune,8000
Mumbai,4000
Jaipur,5000
Chennai,4500


In [0]:
report = (
    customers.join(
        sales,
        "customer_id"
    )
    .groupBy(
        "customer_id",
        "name",
        "city"
    )
    .agg(
        sum("order_amount").alias("total_spend")
    )
)

display(report)

customer_id,name,city,total_spend
C103,Vihaan Kapoor,Delhi,2000
C108,Anika Reddy,Hyderabad,12000
C101,Aadhvik Rao,Bengaluru,6000
C107,Reyansh Gupta,Pune,8000
C102,Ishani Mehta,Mumbai,4000
C110,Tara Malhotra,Jaipur,5000
C106,Kiara Thomas,Chennai,4500
C104,Myra Nair,Hyderabad,9000


In [0]:
display(

report.groupBy(
"city"
)

.agg(
max("total_spend")
.alias("highest_spend")
)

)

city,highest_spend
Delhi,2000
Hyderabad,12000
Bengaluru,6000
Pune,8000
Mumbai,4000
Jaipur,5000
Chennai,4500


5. Build final reporting table with customer, city, total spend, order count

Final ETL Pipeline

In [0]:
%sql

SELECT
    c.customer_id,
    c.name,
    c.city,
    SUM(s.order_amount) AS total_spend,
    COUNT(s.order_id) AS order_count

FROM customers c

JOIN sales s

ON c.customer_id = s.customer_id

GROUP BY
    c.customer_id,
    c.name,
    c.city

customer_id,name,city,total_spend,order_count
C103,Vihaan Kapoor,Delhi,2000,1
C108,Anika Reddy,Hyderabad,12000,3
C101,Aadhvik Rao,Bengaluru,6000,2
C107,Reyansh Gupta,Pune,8000,2
C102,Ishani Mehta,Mumbai,4000,1
C110,Tara Malhotra,Jaipur,5000,1
C106,Kiara Thomas,Chennai,4500,1
C104,Myra Nair,Hyderabad,9000,2


In [0]:
from pyspark.sql.functions import sum, count, col

final_report = (
    customers.join(
        sales,
        "customer_id"
    )
    .groupBy(
        "customer_id",
        "name",
        "city"
    )
    .agg(
        sum("order_amount").alias("total_spend"),
        count("order_id").alias("order_count")
    )
)

display(final_report)

customer_id,name,city,total_spend,order_count
C103,Vihaan Kapoor,Delhi,2000,1
C108,Anika Reddy,Hyderabad,12000,3
C101,Aadhvik Rao,Bengaluru,6000,2
C107,Reyansh Gupta,Pune,8000,2
C102,Ishani Mehta,Mumbai,4000,1
C110,Tara Malhotra,Jaipur,5000,1
C106,Kiara Thomas,Chennai,4500,1
C104,Myra Nair,Hyderabad,9000,2
